In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from ydata_profiling import ProfileReport
from sklearn.preprocessing import LabelEncoder
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.stats import chi2_contingency
import plotly.express as px
import itertools
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, precision_score, recall_score, f1_score
import plotly.graph_objects as go
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import classification_report
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from imblearn.pipeline import Pipeline
from catboost import CatBoostClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import label_binarize
import hashlib
import time
from sklearn.inspection import permutation_importance
from imblearn.over_sampling import SMOTENC
from time import perf_counter
import shap

# Предобработка

In [ ]:
df_cs = pd.read_excel('Мощность.xlsx', sheet_name='кс')
df_cs.head()

In [ ]:
df_cs.info()

In [ ]:
for col in df_cs.columns:
    try:
        uniques = df_cs[col].unique()
    except TypeError:
        uniques = df_cs[col].apply(lambda x: tuple(x) if isinstance(x, (list)) else x).unique()
    print(f'\n{col} ({len(uniques)} уникальных): {uniques}')

In [ ]:
df_cs.isna().sum()

In [ ]:
df_cs.duplicated().sum()

In [ ]:
today = pd.to_datetime("2023-01-01")

df_cs["Лет в эксплуатации"] = (
    today.year - df_cs["Дата ввода в экспл.в БУ"].dt.year
    - ((today.month < df_cs["Дата ввода в экспл.в БУ"].dt.month) |
       ((today.month == df_cs["Дата ввода в экспл.в БУ"].dt.month) &
        (today.day < df_cs["Дата ввода в экспл.в БУ"].dt.day)))
)

df_cs[["Дата ввода в экспл.в БУ", "Лет в эксплуатации"]].head()

In [ ]:
df_cs = df_cs.drop(columns=['Дата ввода в экспл.в БУ', 'Инвентарный номер'])

In [ ]:
df_cs.columns = ['класс', 'эксплуатирующая организация', 'субъект РФ', 'стоимость в ценах 2023 года без НДС, руб', 'мощность, кв', 'лет в эксплуатации']
df_cs.head()

In [ ]:
df_cs = df_cs.apply(
    lambda col: col.astype(str).str.strip().replace(r'\s+', ' ', regex=True)
    if col.dtype == 'object' else col
)

In [ ]:
df_des = pd.read_excel('Мощность.xlsx', sheet_name='дэс')
df_des.head()

In [ ]:
df_des.info()

In [ ]:
for col in df_des.columns:
    try:
        uniques = df_des[col].unique()
    except TypeError:
        uniques = df_des[col].apply(lambda x: tuple(x) if isinstance(x, (list)) else x).unique()
    print(f'\n{col} ({len(uniques)} уникальных): {uniques}')

In [ ]:
df_des.isna().sum()

In [ ]:
df_des.duplicated().sum()

In [ ]:
today = pd.to_datetime("2023-01-01")

df_des["Лет в эксплуатации"] = (
    today.year - df_des["Дата ввода в экспл.в БУ"].dt.year
    - ((today.month < df_des["Дата ввода в экспл.в БУ"].dt.month) |
       ((today.month == df_des["Дата ввода в экспл.в БУ"].dt.month) &
        (today.day < df_des["Дата ввода в экспл.в БУ"].dt.day)))
)

df_des[["Дата ввода в экспл.в БУ", "Лет в эксплуатации"]].head()

In [ ]:
df_des = df_des.drop(columns=['Дата ввода в экспл.в БУ', 'Инвентарный номер'])

In [ ]:
df_des.columns = ['класс', 'эксплуатирующая организация', 'субъект РФ', 'стоимость в ценах 2023 года без НДС, руб', 'мощность, кв', 'лет в эксплуатации']
df_des.head()

In [ ]:
df_des = df_des.apply(
    lambda col: col.astype(str).str.strip().replace(r'\s+', ' ', regex=True)
    if col.dtype == 'object' else col
)

In [ ]:
df_dcs = pd.read_excel('Мощность.xlsx', sheet_name='дкс')
df_des.head()

In [ ]:
df_dcs.info()

In [ ]:
for col in df_dcs.columns:
    try:
        uniques = df_dcs[col].unique()
    except TypeError:
        uniques = df_dcs[col].apply(lambda x: tuple(x) if isinstance(x, (list)) else x).unique()
    print(f'\n{col} ({len(uniques)} уникальных): {uniques}')

In [ ]:
df_dcs.isna().sum()

In [ ]:
df_dcs["Мощность, кВ"].describe()

In [ ]:
df_dcs["Мощность, кВ"] = df_dcs.groupby(["Эксплуатирующая оргпнизация", "Субъект Федерации"])["Мощность, кВ"].transform(
    lambda x: x.replace(0, pd.NA)  
).transform(
    lambda x: x.fillna(x.median())
)

In [ ]:
df_dcs["Мощность, кВ"].describe()

In [ ]:
df_dcs.isna().sum()

In [ ]:
df_dcs.duplicated().sum()

In [ ]:
today = pd.to_datetime("2023-01-01")

df_dcs["Лет в эксплуатации"] = (
    today.year - df_dcs["Дата ввода в экспл.в БУ"].dt.year
    - ((today.month < df_dcs["Дата ввода в экспл.в БУ"].dt.month) |
       ((today.month == df_dcs["Дата ввода в экспл.в БУ"].dt.month) &
        (today.day < df_dcs["Дата ввода в экспл.в БУ"].dt.day)))
)

df_dcs[["Дата ввода в экспл.в БУ", "Лет в эксплуатации"]].head()

In [ ]:
df_dcs = df_dcs.drop(columns=['Дата ввода в экспл.в БУ', 'Инвентарный номер'])

In [ ]:
df_dcs.columns = ['класс', 'эксплуатирующая организация', 'субъект РФ', 'стоимость в ценах 2023 года без НДС, руб', 'мощность, кв', 'лет в эксплуатации']
df_dcs.head()

In [ ]:
df_dcs = df_dcs.apply(
    lambda col: col.astype(str).str.strip().replace(r'\s+', ' ', regex=True)
    if col.dtype == 'object' else col
)

In [ ]:
df_tp = pd.read_excel('Мощность.xlsx', sheet_name='тп')
df_tp.head()

In [ ]:
df_tp.info()

In [ ]:
for col in df_tp.columns:
    try:
        uniques = df_tp[col].unique()
    except TypeError:
        uniques = df_tp[col].apply(lambda x: tuple(x) if isinstance(x, (list)) else x).unique()
    print(f'\n{col} ({len(uniques)} уникальных): {uniques}')

In [ ]:
df_tp.isna().sum()

In [ ]:
df_tp.duplicated().sum()

In [ ]:
today = pd.to_datetime("2023-01-01")

df_tp["Лет в эксплуатации"] = (
    today.year - df_tp["Дата ввода"].dt.year
    - ((today.month < df_tp["Дата ввода"].dt.month) |
       ((today.month == df_tp["Дата ввода"].dt.month) &
        (today.day < df_tp["Дата ввода"].dt.day)))
)

df_tp[["Дата ввода", "Лет в эксплуатации"]].head()

In [ ]:
df_tp = df_tp.drop(columns=['Дата ввода', 'Наименование объекта'])

In [ ]:
df_tp.columns = ['класс', 'эксплуатирующая организация', 'субъект РФ', 'стоимость в ценах 2023 года без НДС, руб', 'мощность, кв', 'лет в эксплуатации']
df_tp.head()

In [ ]:
df_tp = df_tp.apply(
    lambda col: col.astype(str).str.strip().replace(r'\s+', ' ', regex=True)
    if col.dtype == 'object' else col
)

# Объединение данных

In [ ]:
df_power = pd.concat([df_cs, df_des, df_dcs, df_tp], ignore_index=True)
df_power.head()

In [ ]:
df_power.info()

In [ ]:
for col in df_power.columns:
    try:
        uniques = df_power[col].unique()
    except TypeError:
        uniques = df_power[col].apply(lambda x: tuple(x) if isinstance(x, (list)) else x).unique()
    print(f'\n{col} ({len(uniques)} уникальных): {uniques}')

In [ ]:
# Справочник «субъект РФ → код» хранится отдельно и не публикуется, чтобы коды на графиках нельзя было расшифровать.
import json
region_mapping = json.load(open("data/region_mapping.json", encoding="utf-8"))

In [ ]:
df_power['субъект РФ'] = df_power['субъект РФ'].map(region_mapping)

In [ ]:
df_power['субъект РФ'].unique()

In [ ]:
# Справочник «организация → код» не публикуется: названия эксплуатирующих организаций — внутренние данные.
import json
org_mapping = json.load(open("data/org_mapping.json", encoding="utf-8"))

In [ ]:
df_power["эксплуатирующая организация"] = df_power["эксплуатирующая организация"].map(org_mapping)

In [ ]:
df_power["эксплуатирующая организация"].unique()

In [ ]:
df_power.info()

In [ ]:
for col in df_power.columns:
    try:
        uniques = df_power[col].unique()
    except TypeError:
        uniques = df_power[col].apply(lambda x: tuple(x) if isinstance(x, (list)) else x).unique()
    print(f'\n{col} ({len(uniques)} уникальных): {uniques}')

In [ ]:
target_power = 'класс'

target_power_mapping = {
    'Компрессорные станции': 0,
    'Дизельные электростанции': 1,
    'ДКС': 2,
    'Трансформаторные подстанции': 3
}

df_power[target_power] = df_power[target_power].map(target_power_mapping)
df_power.head()

In [ ]:
df_power.to_excel('data/Общий файл (мощность).xlsx', index=False)

In [ ]:
df_power = pd.read_excel("Общий файл (мощность).xlsx")
df_power.head()

In [ ]:
df_power.info()

In [ ]:
X_power = df_power.drop(columns=[target_power])
y_power = df_power[target_power]

# Исследовательский анализ

In [ ]:
num_cols_power = ["стоимость в ценах 2023 года без НДС, руб", "мощность, кв", "лет в эксплуатации"]

plt.figure(figsize=(5, len(num_cols_power) * 3))

for i, col in enumerate(num_cols_power, 1):
    plt.subplot(len(num_cols_power), 1, i)
    sns.histplot(df_power[col], kde=True)
    plt.title(f'Распределение {col}')
    plt.xlabel(col)
    plt.ylabel('Количество')

plt.tight_layout()
plt.show()

In [ ]:
num_cols_power = ["стоимость в ценах 2023 года без НДС, руб", "мощность, кв", "лет в эксплуатации"]

plt.figure(figsize=(20, 20))
sns.pairplot(df_power[num_cols_power], diag_kind="kde")
plt.suptitle("Диаграмма рассеяния", y=1.02)
plt.show()

In [ ]:
num_cols_power = [
    "стоимость в ценах 2023 года без НДС, руб",
    "мощность, кв",
    "лет в эксплуатации"
]

df_corr = df_power[num_cols_power].copy()

rename_dict = {
    "стоимость в ценах 2023 года без НДС, руб": "Стоимость",
    "протяженность, м": "Протяженность",
    "мощность, кв": "Мощность",
    "лет в эксплуатации": "Возраст"
}

df_corr = df_corr.rename(columns=rename_dict)

corr_spearman = df_corr.corr(method="spearman")

plt.figure(figsize=(8, 6))
sns.heatmap(corr_spearman, annot=True, cmap="coolwarm", fmt=".3f")
plt.title("Корреляционная матрица")
plt.show()

In [ ]:
plt.figure(figsize=(8, 10))

plt.subplot(2, 1, 1)
top10_org_power = df_power['эксплуатирующая организация'].value_counts().nlargest(5).index
sns.countplot(x=df_power['эксплуатирующая организация'], order=top10_org_power)
plt.title("Распределение эксплуатирующих организаций (топ 5)")
plt.ylabel('Количество')

plt.subplot(2, 1, 2)
top10_subject_power = df_power['субъект РФ'].value_counts().nlargest(5).index
sns.countplot(x=df_power['субъект РФ'], order=top10_subject_power)
plt.title("Распределение по субъектам (топ 5)")
plt.ylabel('Количество')

plt.tight_layout()
plt.show()

In [ ]:
profile_power = ProfileReport(df_power, title="EDA отчёт", explorative=True)
report_power = ProfileReport(df_power, title="Первичный исследовательский анализ", explorative=True)
profile_power.to_file('data/eda_power_report.html')
report_power.to_notebook_iframe()

# Обучение

In [ ]:
num_power_features = X_power.select_dtypes(include=['int32', 'int64', 'float64']).columns.tolist()
cat_power_features = X_power.select_dtypes(include=['object', 'category']).columns.tolist()

numerical_power_transformer = StandardScaler()
categorical_power_transformer = OneHotEncoder(handle_unknown='ignore')
preprocessor_power = ColumnTransformer(
    transformers=[
        ('num', numerical_power_transformer, num_power_features),
        ('cat', categorical_power_transformer, cat_power_features)
    ])

In [ ]:
X_power_train, X_power_test, y_power_train, y_power_test = train_test_split(
    X_power, y_power, test_size=0.2, random_state=123, stratify=y_power
)

In [ ]:
logreg_power = ImbPipeline([
    ('preprocessor', preprocessor_power),
    ('smote', SMOTE(random_state=123)),
    ('classifier', LogisticRegression(
        multi_class='multinomial',
        class_weight='balanced',
        max_iter=1000
    ))
])

param_grid_logreg_power = {
    'classifier__solver': ['liblinear', 'saga'], 
    'classifier__penalty': ['l1', 'l2', 'elasticnet'],  
    'classifier__C': [0.1, 1, 10],  
    'classifier__l1_ratio': [0, 0.5, 0.9],  
    'smote__k_neighbors': [2, 3, 5]  
}

cv_logreg_power = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

grid_logreg_power = GridSearchCV(
    estimator=logreg_power,
    param_grid=param_grid_logreg_power,
    scoring='f1_macro',
    cv=cv_logreg_power,
    n_jobs=-1,
    verbose=1
)

grid_logreg_power.fit(X_power_train, y_power_train)

In [ ]:
power_colors = {
    0: "#1f77b4",  # синий — компрессорные станции
    1: "#ff7f0e",  # оранжевый — дизельные электростанции
    2: "#2ca02c",  # зелёный — ДКС
    3: "#d62728"   # красный — ТП
}

power_class_names = {
    0: 'КС',
    1: 'ДЭС',
    2: 'ДКС',
    3: 'ТП'
}

In [ ]:
print("Лучшие параметры:", grid_logreg_power.best_params_)
print("Лучший ROC-AUC (train):", grid_logreg_power.best_score_)

y_logreg_power_pred_proba = grid_logreg_power.predict_proba(X_power_test)

roc_logreg_power_test = roc_auc_score(
    y_power_test,
    y_logreg_power_pred_proba,
    multi_class='ovr'
)
print("ROC-AUC (test):", roc_logreg_power_test)

y_logreg_power_pred = grid_logreg_power.predict(X_power_test)

cm = confusion_matrix(y_power_test, y_logreg_power_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=list(power_class_names.values())
)

plt.figure(figsize=(8, 6))
disp.plot(
    cmap='Blues',
    values_format='d',
    colorbar=False
)
plt.title("Матрица ошибок Logistic Regression", fontsize=14)

print(classification_report(y_power_test, y_logreg_power_pred))

In [ ]:
classes = [0, 1, 2, 3]

y_test_bin = label_binarize(y_power_test, classes=classes)
proba = grid_logreg_power.predict_proba(X_power_test)

fig = go.Figure()

for class_idx, class_name in power_class_names.items():
    fpr, tpr, _ = roc_curve(y_test_bin[:, class_idx], proba[:, class_idx])
    auc = roc_auc_score(y_test_bin[:, class_idx], proba[:, class_idx])
    
    fig.add_trace(go.Scatter(
        x=fpr, 
        y=tpr,
        mode="lines",
        name=f"{class_name} (AUC={auc:.3f})",
        line=dict(width=3, color=power_colors[class_idx])
    ))

fig.add_trace(go.Scatter(
    x=[0,1], y=[0,1], 
    mode="lines",
    name="Random baseline",
    line=dict(dash="dash", color="gray")
))

fig.update_layout(
    title="OvR ROC-кривые Logistic Regression",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    template="plotly_white",
    width=900,
    height=600,
    font=dict(size=13)
)

fig.show()

In [ ]:
knn_power_power = ImbPipeline([
    ('preprocessor', preprocessor_power),
    ('smote', SMOTE(random_state=123)),
    ('classifier', KNeighborsClassifier())
])

param_grid_knn_power = {
    'classifier__n_neighbors': [3, 5, 7, 10],
    'classifier__weights': ['uniform', 'distance'],
    'classifier__metric': ['euclidean', 'manhattan'],
    'smote__k_neighbors': [2, 3, 5]
}
cv_knn_power = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

grid_knn_power = GridSearchCV(
    knn_power_power,
    param_grid_knn_power,
    cv=cv_knn_power,
    scoring='roc_auc_ovr',
    n_jobs=-1,
    verbose=1
)

grid_knn_power.fit(X_power_train, y_power_train)

In [ ]:
print("Лучшие параметры KNN:", grid_knn_power.best_params_)
print("Лучший ROC-AUC (train):", grid_knn_power.best_score_)

y_knn_power_pred = grid_knn_power.predict(X_power_test)
y_knn_power_pred_proba = grid_knn_power.predict_proba(X_power_test)

roc_knn_power_test = roc_auc_score(
    y_power_test,
    y_knn_power_pred_proba,
    multi_class='ovr'
)
print("ROC-AUC (test):", roc_knn_power_test)

custom_labels = ["КС", "ДЭС", "ДКС", "ТП"]
cm = confusion_matrix(y_power_test, y_knn_power_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=custom_labels)

disp.plot(cmap='Blues', values_format='d')
plt.title("Матрица ошибок KNN")
plt.show()

print(classification_report(y_power_test, y_knn_power_pred))

In [ ]:
classes = [0, 1, 2, 3]

y_test_bin = label_binarize(y_power_test, classes=classes)
proba = y_knn_power_pred_proba

fig = go.Figure()

for i, name in power_class_names.items():
    frp, tpr, _ = roc_curve(y_test_bin[:, i], proba[:, i])
    auc = roc_auc_score(y_test_bin[:, i], proba[:, i])
    
    fig.add_trace(go.Scatter(
        x=frp,
        y=tpr,
        mode="lines",
        name=f"{name} (AUC={auc:.3f})",
        line=dict(width=3)
    ))

fig.add_trace(go.Scatter(
    x=[0,1], y=[0,1], mode="lines", 
    name="Random", 
    line=dict(color='gray', dash="dash")
))

fig.update_layout(
    title="OvR ROC-кривые KNN",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    template="plotly_white",
    width=900,
    height=600
)

fig.show()

In [ ]:
dt_power = Pipeline(steps=[
    ('preprocessor', preprocessor_power),
    ('smote', SMOTE(random_state=123)),
    ('classifier', DecisionTreeClassifier(random_state=123))
])

param_grid_dt_power = {
    "classifier__max_depth": [5, 10, 15],
    "classifier__min_samples_split": [5, 10, 15],
    "classifier__min_samples_leaf": [2, 4, 6],
    "classifier__criterion": ["gini", "entropy"],
    "smote__k_neighbors": [2, 3, 5]
}

cv_dt_power = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

grid_dt_power = GridSearchCV(
    dt_power,
    param_grid_dt_power,
    scoring='roc_auc_ovr',      
    cv=cv_dt_power,
    n_jobs=-1,
    verbose=1
)

grid_dt_power.fit(X_power_train, y_power_train)

In [ ]:
print("Лучшие параметры:", grid_dt_power.best_params_)
y_dt_power_pred = grid_dt_power.predict(X_power_test)
y_dt_power_pred_proba = grid_dt_power.predict_proba(X_power_test)

roc_dt_power_test = roc_auc_score(
    y_power_test,
    y_dt_power_pred_proba,
    multi_class="ovr"
)

print("ROC-AUC (test, OVR):", roc_dt_power_test)
class_names = [
    'КС',
    'ДЭС',
    'ДКС',
    'ТП'
]
cm = confusion_matrix(y_power_test, y_dt_power_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap='Blues')
plt.title("Матрица ошибок Decision Tree")
plt.show()

print(classification_report(y_power_test, y_dt_power_pred))

In [ ]:
classes = [0, 1, 2, 3]
y_test_bin = label_binarize(y_power_test, classes=classes)

fig = go.Figure()

class_names = [
    'Компрессорные станции',
    'Дизельные электростанции',
    'ДКС',
    'Трансформаторные подстанции'
]

for i, cname in enumerate(class_names):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_dt_power_pred_proba[:, i])
    auc = roc_auc_score(y_test_bin[:, i], y_dt_power_pred_proba[:, i])
    
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr, mode="lines",
        name=f"{cname} (AUC={auc:.3f})"
    ))

fig.add_trace(go.Scatter(
    x=[0,1], y=[0,1], mode="lines",
    name="Random", line=dict(dash="dash")
))

fig.update_layout(
    title='OvR ROC-кривые Decision Tree',
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    template="plotly_white",
    height=750,
    width=750,
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.2,
        xanchor="center",
        x=0.5
    )
)

fig.show()

In [ ]:
num_features = num_power_features

cat_encoder = preproc.named_transformers_['cat']
cat_features = cat_encoder.get_feature_names_out(cat_power_features)

feature_names = np.concatenate([num_features, cat_features])

X_transformed = preproc.transform(X_power_test)

if hasattr(X_transformed, "toarray"):
    X_transformed = X_transformed.toarray()

X_transformed = pd.DataFrame(
    X_transformed,
    columns=feature_names)


explainer = shap.TreeExplainer(clf)

shap_values = explainer.shap_values(X_transformed)

shap_vals = shap_values[:, :, 1]

In [ ]:
scaler = StandardScaler()

X_train_scaled = X_power_train.copy()
X_test_scaled = X_power_test.copy()

X_train_scaled[num_power_features] = scaler.fit_transform(
    X_train_scaled[num_power_features]
)

X_test_scaled[num_power_features] = scaler.transform(
    X_test_scaled[num_power_features]
)

for c in cat_power_features:
    X_train_scaled[c] = X_train_scaled[c].astype(str)
    X_test_scaled[c] = X_test_scaled[c].astype(str)
    
cat_idx = [X_train_scaled.columns.get_loc(c) for c in cat_power_features]

smote_nc = SMOTENC(
    categorical_features=cat_idx,
    random_state=123
)

X_res, y_res = smote_nc.fit_resample(
    X_train_scaled,
    y_power_train
)

model_cb_power = CatBoostClassifier(
    random_state=123,
    loss_function="MultiClass",
    eval_metric="AUC",
    verbose=0
)

param_grid_cb_power = {
    "depth": [3, 5, 10],
    "learning_rate": [0.03, 0.05],
    "iterations": [100, 200, 300],
    "l2_leaf_reg": [2, 3, 5]
}

cv_cb_power = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)

grid_cb_power = GridSearchCV(
    estimator=model_cb_power,
    param_grid=param_grid_cb_power,
    scoring="roc_auc_ovr_weighted",
    cv=cv_cb_power,
    n_jobs=-1,
    verbose=1
)

grid_cb_power.fit(
    X_res,
    y_res,
    cat_features=cat_idx
)

In [ ]:
print("Лучшие параметры CatBoost:", grid_cb_power.best_params_)
print("Лучший ROC-AUC (train):", grid_cb_power.best_score_)

proba_power_cb = grid_cb_power.predict_proba(X_test_scaled)
roc_power_cb_test = roc_auc_score(
    y_power_test,
    proba_power_cb,
    multi_class="ovr"
)
print("ROC-AUC (test):", roc_power_cb_test)

y_pred_power_cb = grid_cb_power.predict(X_test_scaled)

cm = confusion_matrix(y_power_test, y_pred_power_cb)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=list(target_power_mapping.keys())
)

disp.plot(
    cmap="Blues",
    values_format="d"
)
plt.title("Матрица ошибок CatBoost")
plt.show()

In [ ]:
classes = [0, 1, 2, 3]
y_test_bin = label_binarize(y_power_test, classes=classes)

fig = go.Figure()

names = ["Компрессорные станции", "Дизельные электростанции", "ДКС", "ТП"]

for i, class_name in enumerate(names):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], proba_power_cb[:, i])
    auc = roc_auc_score(y_test_bin[:, i], proba_power_cb[:, i])
    
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr, mode="lines",
        name=f"{class_name} (AUC={auc:.3f})",
        line=dict(width=3)
    ))

fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode="lines",
    name="Random",
    line=dict(dash="dash", color="gray")
))

fig.update_layout(
    title="OvR ROC-кривые CatBoost",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    template="plotly_white",
    width=900,
    height=600
)

fig.show()

In [ ]:
results = []

def evaluate_model(
    name,
    model,
    X_train,
    y_train,
    X_test,
    y_test,
    cat_features=None
):
    start = perf_counter()

    if cat_features is not None:
        model.fit(X_train, y_train, cat_features=cat_features)
    else:
        model.fit(X_train, y_train)

    train_time = perf_counter() - start

    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)
    else:
        y_proba = None

    auc = None
    if y_proba is not None:
        if y_proba.ndim == 1 or y_proba.shape[1] == 1:
            auc = roc_auc_score(y_test, y_proba)
        else:
            auc = roc_auc_score(
                y_test,
                y_proba,
                multi_class="ovr",
                average="weighted"
            )

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    results.append({
        "model": name,
        "train_time_sec": round(train_time, 3),
        "AUC": auc,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1
    })

In [ ]:
evaluate_model("Logistic Regression", grid_logreg_power, X_power_train, y_power_train, X_power_test, y_power_test)
evaluate_model("KNN", grid_knn_power, X_power_train, y_power_train, X_power_test, y_power_test)
evaluate_model("Decision Tree", grid_dt_power, X_power_train, y_power_train, X_power_test, y_power_test)
evaluate_model(
    name="CatBoost",
    model=grid_cb_power,
    X_train=X_res,
    y_train=y_res,
    X_test=X_test_scaled,
    y_test=y_power_test,
    cat_features=cat_idx
)

In [ ]:
df_metrics = pd.DataFrame(results)
df_metrics

In [ ]:
feature_names = (
    grid_logreg_power.best_estimator_["preprocessor"]
    .get_feature_names_out()
)

coefs_logreg = np.abs(grid_logreg_power.best_estimator_["classifier"].coef_).mean(axis=0)

fi_logreg = pd.DataFrame({
    "feature": feature_names,
    "importance": coefs_logreg
}).sort_values("importance", ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(fi_logreg["feature"].head(10), fi_logreg["importance"].head(10))
plt.title("Важность признаков - Logistic Regression")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
best_model = grid_knn_power.best_estimator_

preproc = best_model.named_steps["preprocessor"]
preproc.fit(X_power_train)

X_test_transformed = preproc.transform(X_power_test)

scoring_metric = "roc_auc" if len(np.unique(y_power_test)) == 2 else "roc_auc_ovr"

X_test_dense = X_test_transformed.toarray()

r = permutation_importance(
    estimator=best_model.named_steps["classifier"],  
    X=X_test_dense,
    y=y_power_test,
    n_repeats=10,
    random_state=123,
    scoring=scoring_metric
)

feature_names = preproc.get_feature_names_out()

fi = pd.DataFrame({
    "feature": feature_names,
    "importance": r.importances_mean
}).sort_values("importance", ascending=False)

plt.figure(figsize=(8, 8))
plt.barh(fi["feature"].head(10), fi["importance"].head(10))
plt.title("Пермутационная важность признаков — KNN")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
best_tree = grid_dt_power.best_estimator_
clf = best_tree.named_steps["classifier"]
preproc = best_tree.named_steps['preprocessor']

best_tree = grid_dt_power.best_estimator_
clf = best_tree.named_steps["classifier"]
preproc = best_tree.named_steps['preprocessor']

feature_names = preproc.get_feature_names_out()
importances = clf.feature_importances_

fi = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

plt.figure(figsize=(8, 6))
plt.barh(fi["feature"].head(10), fi["importance"].head(10))
plt.title("Важность признаков — Decision Tree")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
for class_id in range(4):
    shap.summary_plot(
        shap_values[:, :, class_id],
        X_transformed,
        plot_type="bar",
        show=True
    )

In [ ]:
for class_idx in range(4):
    shap.summary_plot(
        shap_values[:, :, class_idx],
        X_transformed,
        show=True,
        plot_type="dot"
    )

In [ ]:
clf = grid_cb_power.best_estimator_
importances = clf.get_feature_importance()
feature_names = X_train_scaled.columns
fi = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)
plt.figure(figsize=(8, 6))
plt.barh(fi["feature"], fi["importance"])
plt.title("Важность признаков — CatBoost")
plt.gca().invert_yaxis()
plt.show()